# MFCAD Dataset Exploration
**Author:** Eduardo Dall'Igna  
**Project:** CAD Model Retrieval via Unsupervised Graph Learning  
**Course:** Software Lab 2026 — TUM

This notebook explores the structure and statistics of the preprocessed MFCAD dataset (.pt graph files) to support evaluation design and presentation insights.

## 1. Setup

In [ ]:
import torch
import glob
import os
import statistics
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter

# Path to processed graphs
DATA_DIR = '../data/processed_graphs/mfcad'
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.pt')))
names = [os.path.basename(f).replace('.pt', '') for f in files]

print(f'Total .pt files found: {len(files)}')

## 2. Single Graph Inspection
Understanding what one CAD model looks like as a graph.

In [ ]:
sample = torch.load(files[0], weights_only=False)

print('=== Sample CAD Graph ===')
print(f'File:              {names[0]}')
print(f'Nodes (faces):     {sample.x.shape[0]}')
print(f'Node features:     {sample.x.shape[1]}  -> [area, surface_type, closed_u, closed_v]')
print(f'Edges (directed):  {sample.edge_index.shape[1]}  ({sample.edge_index.shape[1]//2} unique shared edges)')
print(f'Edge features:     {sample.edge_attr.shape[1]}  -> [length, curve_type, closed]')
print()
print('Node feature matrix (first 5 nodes):')
print(sample.x[:5])

## 3. Graph Size Statistics
How complex are the CAD models in terms of graph size?

In [ ]:
node_counts = []
edge_counts = []

for f in files:
    data = torch.load(f, weights_only=False)
    node_counts.append(data.x.shape[0])
    edge_counts.append(data.edge_index.shape[1] // 2)

print('=== Graph Size Statistics ===')
print(f'Avg faces per model:  {statistics.mean(node_counts):.1f}')
print(f'Median faces:         {statistics.median(node_counts):.1f}')
print(f'Min faces:            {min(node_counts)}')
print(f'Max faces:            {max(node_counts)}')
print()
print(f'Avg edges per model:  {statistics.mean(edge_counts):.1f}')
print(f'Min edges:            {min(edge_counts)}')
print(f'Max edges:            {max(edge_counts)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(node_counts, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Faces per CAD Model', fontsize=13)
axes[0].set_xlabel('Number of Faces (Nodes)')
axes[0].set_ylabel('Count')
axes[0].axvline(statistics.mean(node_counts), color='red', linestyle='--', label=f'Mean: {statistics.mean(node_counts):.1f}')
axes[0].legend()

axes[1].hist(edge_counts, bins=20, color='darkorange', edgecolor='white')
axes[1].set_title('Distribution of Edges per CAD Model', fontsize=13)
axes[1].set_xlabel('Number of Unique Edges')
axes[1].set_ylabel('Count')
axes[1].axvline(statistics.mean(edge_counts), color='red', linestyle='--', label=f'Mean: {statistics.mean(edge_counts):.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../notebooks/fig_graph_size_distribution.png', dpi=150)
plt.show()
print('Saved: fig_graph_size_distribution.png')

## 4. Class Label Analysis
MFCAD filenames encode a 2-level class hierarchy: `TopClass-SubClass-...-InstanceID`

In [ ]:
top_labels = [n.split('-')[0] for n in names]
second_labels = [f"{n.split('-')[0]}-{n.split('-')[1]}" for n in names]

top_counts = Counter(top_labels)
second_counts = Counter(second_labels)

print(f'Top-level classes:    {len(top_counts)}')
print(f'Second-level classes: {len(second_counts)}')
print()
print('Top-level class distribution:')
for cls, count in sorted(top_counts.items(), key=lambda x: int(x[0])):
    print(f'  Class {cls:>2}: {count:>5} models')

In [ ]:
sorted_classes = sorted(top_counts.items(), key=lambda x: int(x[0]))
labels = [f'Class {c}' for c, _ in sorted_classes]
values = [v for _, v in sorted_classes]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color='steelblue', edgecolor='white')

# Annotate bars
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(val), ha='center', va='bottom', fontsize=9)

ax.set_title('Class Imbalance in MFCAD Dataset (Top-Level)', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Number of Models')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../notebooks/fig_class_distribution.png', dpi=150)
plt.show()
print('Saved: fig_class_distribution.png')

## 5. Class Imbalance — Key Insight

The dataset is **heavily imbalanced**:
- Class 0 dominates with ~3,875 models
- Classes 12–14 have fewer than 35 models each

**Implication for retrieval evaluation:** Recall@K metrics will be biased toward majority classes. The model may appear to perform well overall while completely failing on rare classes. This is a likely contributor to the poor results observed in the ablation study.

## 6. Node Feature Statistics
Understanding the raw geometric features across the dataset.

In [ ]:
all_node_features = []

for f in files:
    data = torch.load(f, weights_only=False)
    all_node_features.append(data.x.numpy())

all_features = np.vstack(all_node_features)
feature_names = ['Area', 'Surface Type', 'Closed U', 'Closed V']

print('=== Node Feature Statistics (across all faces in dataset) ===')
for i, name in enumerate(feature_names):
    col = all_features[:, i]
    print(f'{name:>15}: min={col.min():.3f}, max={col.max():.3f}, mean={col.mean():.3f}, std={col.std():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (name, ax) in enumerate(zip(feature_names, axes)):
    col = all_features[:, i]
    if name == 'Area':
        # Clip outliers for readability
        col = col[col < np.percentile(col, 99)]
    ax.hist(col, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.suptitle('Node Feature Distributions (All Faces in MFCAD)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../notebooks/fig_feature_distributions.png', dpi=150)
plt.show()
print('Saved: fig_feature_distributions.png')

## 7. Summary of Findings

| Finding | Value | Implication |
|---|---|---|
| Total models | 15,488 | Reasonable training set size |
| Avg faces per model | ~18 | Small graphs — GNN should converge fast |
| Face range | 8–33 | Low variance in complexity |
| Top-level classes | 15 | Coarse label available for evaluation |
| Second-level classes | 120 | Fine-grained label available |
| Class imbalance ratio | ~969x (Class 0 vs 14) | Major evaluation bias risk |
| Area feature | High variance, skewed | Needs normalization before training |